# UD2 · Práctica · pandas: leer formatos, explorar y resumir

**MP5074 Sistemas de Big Data**

Trabajamos con los datos de *Tecnoventas*, una cadena ficticia de tiendas de electrónica. La práctica tiene dos partes:

| Parte | Qué practicas | Ficheros |
|---|---|---|
| **A · Leer formatos** | CSV exportado de Windows, JSON anidado, JSON Lines comprimido y Excel con varias hojas | `ventas_2024_tpv.csv`, `catalogo.json`, `visitas_web.jsonl.gz`, `objetivos_2025.xlsx` |
| **B · Explorar y resumir** | Explorar, seleccionar, filtrar, crear columnas, agrupar y guardar | `ventas.csv` |

Todo está explicado en el [manual de pandas](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/) (apartados 3 a 12).

**Cómo trabajar**

- Resuelve cada celda **Tu turno** antes de seguir. Cada ejercicio termina con un bloque **✔ Comprueba** con lo que debes obtener.
- Si algo se rompe: *Kernel → Restart Kernel and Run All Cells*.

> Necesitas `pandas`, `openpyxl` y `pyarrow`: `conda install -c conda-forge pandas openpyxl pyarrow`

**Antes de empezar: los datos.** El cuaderno necesita la carpeta `datos/` **junto a él**, con 5 ficheros
(`ventas.csv`, `ventas_2024_tpv.csv`, `catalogo.json`, `visitas_web.jsonl.gz` y `objetivos_2025.xlsx`).
Están en el repositorio de la web, en [`notebooks/ud2/pandas/`](https://github.com/rafafl-wirtz/mp5074-sistemas-big-data/tree/main/notebooks/ud2/pandas). Para tenerlos:

- **Opción 1 (recomendada):** clona el repositorio (`git clone`) o descárgalo entero (**Code → Download ZIP**) y abre el cuaderno desde `notebooks/ud2/pandas/`.
- **Opción 2:** crea una carpeta `datos/` junto al cuaderno y descarga en ella los 5 ficheros desde GitHub (botón **Download raw file**).

Si falta algún fichero, la celda de preparación te dirá cuál.

## 0. Preparación

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATOS = Path("datos")        # carpeta con los ficheros de la práctica (junto al cuaderno)
SALIDA = Path("salida")      # aquí guardaremos los resultados
SALIDA.mkdir(exist_ok=True)

NECESARIOS = ["ventas.csv", "ventas_2024_tpv.csv", "catalogo.json", "visitas_web.jsonl.gz", "objetivos_2025.xlsx"]
faltan = [f for f in NECESARIOS if not (DATOS / f).exists()]
if faltan:
    raise FileNotFoundError(f"No encuentro {faltan} en {DATOS.resolve()}. "
                            "Descarga la carpeta datos/ junto al cuaderno (instrucciones al principio).")

for f in NECESARIOS:                     # inventario: qué ficheros hay y cuánto ocupan
    print(f"{f:24} {(DATOS / f).stat().st_size / 1024:8.1f} KB")

# Parte A · Leer formatos

## 1. CSV exportado de Windows

Manual: [4.1 CSV](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#41-csv).

Antes de leer un fichero con pandas, **míralo**. El del TPV no es UTF-8, así que lo abrimos con la codificación de Windows (`cp1252`):

In [ ]:
with open(DATOS / "ventas_2024_tpv.csv", encoding="cp1252") as f:
    for _ in range(5):
        print(repr(f.readline()))     # repr muestra también ; \n y espacios

Ya sabemos cuatro cosas **antes** de usar pandas: codificación `cp1252`, **3 líneas decorativas** antes de la cabecera,
separador **`;`** y decimales con **coma**. Si lo leyéramos sin parámetros, fallaría:

In [ ]:
try:
    pd.read_csv(DATOS / "ventas_2024_tpv.csv")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError →", e)

> ### Tu turno · Ejercicio 1
>
> 1. Lee `ventas_2024_tpv.csv` en `tpv` con los parámetros necesarios: `encoding`, `sep`, `skiprows`, `decimal=","` y `thousands="."`.
> 2. Mira las **últimas** filas (`tail`). ¿Qué hay al final que **no** es una venta?
> 3. Comprueba el tipo de `Precio Unit. (€)` y de `Importe (€)`. ¿Por qué uno es número y el otro texto?
>
> **✔ Comprueba:** `tpv.shape` es `(432, 12)`; el precio es `float64` y el importe sigue siendo texto.

In [ ]:
tpv = ...   # Tu código aquí


**Tu respuesta:** …

## 2. JSON anidado: el catálogo

Manual: [4.3 JSON](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#43-json).

Un JSON se explora primero como un diccionario de Python:

In [ ]:
with open(DATOS / "catalogo.json", encoding="utf-8") as f:
    catalogo = json.load(f)

print(catalogo.keys())                                              # metadatos y la lista de productos
print(json.dumps(catalogo["productos"][0], indent=2, ensure_ascii=False))   # un producto, bien indentado

Los registros están en la lista `productos`. Cada producto tiene diccionarios dentro (`proveedor`, `precios`, `stock`) y una lista (`etiquetas`).

> ### Tu turno · Ejercicio 2
>
> 1. Convierte `catalogo["productos"]` en el DataFrame `productos` con `pd.json_normalize(..., sep="_")`.
> 2. Añade la columna `margen` = `precios_pvp` − `precios_coste`.
> 3. Muestra `sku`, `nombre`, `proveedor_nombre` y `margen`, de mayor a menor margen.
>
> **✔ Comprueba:** `productos` tiene `11` filas y el producto con más margen es **PORTATIL**.

In [ ]:
productos = ...   # Tu código aquí


## 3. JSON Lines comprimido: un fichero grande

Manual: [4.3 JSON](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#43-json).

**JSON Lines** (`.jsonl`) tiene un objeto JSON **por línea**: es el formato típico de los registros de actividad (*logs*).
Este además viene comprimido con gzip; pandas lo descomprime solo por la extensión `.gz`.

> ### Tu turno · Ejercicio 3
>
> 1. Lee `visitas_web.jsonl.gz` en `visitas` (parámetro `lines=True`).
> 2. ¿Cuántos eventos hay de cada tipo (`evento`)? ¿Cuántas sesiones distintas (`sesion`)?
> 3. ¿Cuánta memoria ocupa (`memory_usage(deep=True)`) comparada con lo que ocupa en disco?
>
> **✔ Comprueba:** `249662` filas y `55000` sesiones distintas.

In [ ]:
visitas = ...   # Tu código aquí


## 4. Excel con varias hojas

Manual: [4.2 Excel](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#42-excel).

`objetivos_2025.xlsx` tiene **una hoja por región** con los objetivos mensuales de cada vendedor. Cada hoja tiene un título y una fila en blanco antes de la tabla.

> ### Tu turno · Ejercicio 4
>
> 1. Lee **todas** las hojas a la vez con `sheet_name=None` y `skiprows=3`: obtienes un diccionario `{hoja: DataFrame}`.
> 2. Únelas en un solo DataFrame `objetivos` con una columna `region` que diga de qué hoja viene cada fila
>    (recorre el diccionario, añade la columna y usa `pd.concat(lista, ignore_index=True)`).
> 3. Calcula el objetivo **anual** de cada región (suma de los 12 meses).
>
> **✔ Comprueba:** `objetivos` tiene 30 (5 regiones × 6 vendedores) filas y la región con mayor objetivo anual es **C Valenciana**.

In [ ]:
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
hojas = ...   # Tu código aquí


# Parte B · Explorar y resumir `ventas.csv`

## 5. Leer y explorar

Manual: [6. Primera exploración](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#6-primera-exploración).

`ventas.csv` es un CSV limpio (UTF-8, coma, punto decimal). Solo hay que decirle a pandas qué columna es una fecha.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas.csv", parse_dates=["fecha"])
print(ventas.shape)
ventas.info()

**Rutina de las 4 comprobaciones** después de leer cualquier fichero: `shape`, `head()`/`tail()`, `info()`/`dtypes` e `isna().sum()`.

> ### Tu turno · Ejercicio 5
>
> 1. Muestra `describe()` redondeado a 2 decimales. En `total`, ¿la media está cerca de la mediana (50 %)? ¿Qué significa?
> 2. ¿Cuántos vendedores distintos hay? ¿Qué método de pago es el más usado?
> 3. ¿Cuál es la fecha de la primera y de la última venta?
>
> **✔ Comprueba:** `6` vendedores; el método más usado es **Tarjeta**.

In [ ]:
# Tu código aquí


**Tu respuesta:** …

## 6. Seleccionar y filtrar

Manual: [7. Seleccionar y filtrar](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#7-seleccionar-y-filtrar).

| Expresión | Devuelve |
|---|---|
| `df["col"]` / `df[["c1", "c2"]]` | Una columna (`Series`) / varias (`DataFrame`) |
| `df.loc[filas, cols]` | Por **etiqueta**; los rangos **incluyen** el final |
| `df.iloc[filas, cols]` | Por **posición**; los rangos **excluyen** el final |
| `df[máscara]` | Las filas donde la máscara es `True`. Varias condiciones: `&`, `|`, `~` y **paréntesis** |

> ### Tu turno · Ejercicio 6
>
> 1. Con `loc`, muestra `vendedor`, `region` y `total` de las filas con etiqueta 100 a 105.
> 2. ¿Cuántas ventas de **Cataluña** se pagaron con **PayPal**?
> 3. ¿Cuántas ventas de **Portátil** o **Smartphone** tienen **5** unidades? (usa `isin`)
> 4. ¿Cuántas ventas **no** son de la categoría Informática? (usa `~`)
>
> **✔ Comprueba:** 6 filas en el apartado 1; respuestas 2) 30, 3) 24 y 4) 305.

In [ ]:
# Tu código aquí


## 7. Ordenar y crear columnas

Manual: [8. Crear y modificar columnas](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#8-crear-y-modificar-columnas) y [11. Ordenar](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#11-ordenar-y-clasificar).

Las operaciones se aplican a **columnas enteras**, sin bucles. Para fechas existe el accesor `.dt`; para textos, `.str`.

> ### Tu turno · Ejercicio 7
>
> 1. Añade las columnas `con_descuento` (`True` si `descuento > 0`), `trimestre` (`.dt.quarter`) e `iva` (21 % del `total`, 2 decimales).
> 2. ¿Qué porcentaje de ventas tiene descuento? (pista: la media de una columna booleana)
> 3. Muestra las 5 ventas con **más unidades** y, en caso de empate, mayor `total` primero.
>
> **✔ Comprueba:** El 47,8 % de las ventas tienen descuento.

In [ ]:
# Tu código aquí


## 8. Agrupar

Manual: [12. Agrupar y agregar](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#12-agrupar-y-agregar).

`groupby` **divide** las filas en grupos, **aplica** una función a cada uno y **combina** los resultados (como `GROUP BY` en SQL). Con `agg` y nombres se calculan varias cosas a la vez:

```python
df.groupby("clave").agg(nombre_resultado=("columna", "función"), ...)
```

> ### Tu turno · Ejercicio 8
>
> 1. Crea `resumen` con, para cada **región**: número de ventas, unidades, facturación, ticket medio y vendedores distintos. Redondea a 2 decimales y ordena por facturación.
> 2. Facturación por **trimestre** y **canal** (dos claves de agrupación).
>
> **✔ Comprueba:** `resumen` tiene 5 filas y la región que más factura es **C. Valenciana**.

In [ ]:
resumen = ...   # Tu código aquí


## 9. Reto final: encadenar y guardar

Manual: [5. Guardar datos y elegir formato](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#5-guardar-datos-y-elegir-formato).

> ### Tu turno · Ejercicio 9
>
> 1. Con **una sola expresión encadenada** (sin variables intermedias), obtén los **3 productos que más facturan en Galicia por canal Online** → `top3`.
> 2. Guarda `resumen` en `salida/resumen_regiones.csv` (aquí **sí** interesa el índice: es la región) y `ventas` en `salida/ventas.parquet` (`index=False`).
> 3. Vuelve a leer los dos ficheros: ¿qué tipo tiene la columna `fecha` en el Parquet? ¿Y si guardaras `ventas` en CSV y lo leyeras sin `parse_dates`?
>
> **✔ Comprueba:** el primer producto de `top3` es **Smartphone**; en el Parquet `fecha` sigue siendo fecha.

In [ ]:
top3 = ...   # Tu código aquí


**Tu respuesta:** …

## Resumen

| Quiero… | Uso |
|---|---|
| Leer un CSV difícil | `read_csv(encoding=, sep=, skiprows=, decimal=, thousands=, parse_dates=)` |
| Leer JSON anidado | `json.load` + `pd.json_normalize(lista, sep="_")` |
| Leer JSON Lines | `read_json(ruta, lines=True)` |
| Leer todas las hojas de un Excel | `read_excel(ruta, sheet_name=None, skiprows=)` + `concat` |
| Primer vistazo | `shape`, `head`, `info`, `describe`, `value_counts`, `nunique`, `isna().sum()` |
| Seleccionar / filtrar | `loc`, `iloc`, `df[máscara]`, `&`, `|`, `~`, `isin` |
| Columnas nuevas | operaciones vectorizadas, `.dt`, `.str` |
| Resumir por grupos | `groupby(...).agg(nombre=("col", "func"))` |
| Guardar | `to_parquet` (conserva tipos), `to_csv` |

**Siguiente paso:** limpieza de datos ([manual §10](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/pandas/manual_pandas/#10-limpieza-de-datos)).